In [45]:
import pandas as pd
import numpy as np
import glob
import os

# List of transactions to extract
transactions = [
    "Iron and steel",
    "Chemical and petrochemical",
    "Non-ferrous metals",
    "Non-metallic minerals",
    "Transport equipment",
    "Machinery",
    "Mining and quarrying",
    "Food and tobacco",
    "Paper  pulp and printing",
    "Wood and wood products",
    "Textile and leather",
    "Construction",
    "Industry n.e.s"
]

folder_path = r"C:\Users\eduque\Fraunhofer\DeV-KopSys-2 - Globale ESA\04_methods_and_models\Sectoral_End-Use_Energy_Scenario\subsector_workflows\industry\data_preparation\input_data\country_specific"
all_files = glob.glob(os.path.join(folder_path, "*.csv"))

dfs = []
for file in all_files:
    df = pd.read_csv(file, dtype=str)
    df = df.replace('---', np.nan)
    df = df.applymap(lambda x: x.lstrip('*') if isinstance(x, str) else x)
    # Extract only the rows for the specified transactions (case-insensitive, strip spaces)
    mask = df['Transactions(down)/Commodity(right)'].str.strip().str.lower().isin([t.lower() for t in transactions])
    df_transactions = df[mask].copy()
    # Convert all columns except the first two to numeric
    for col in df_transactions.columns[2:]:
        df_transactions[col] = pd.to_numeric(df_transactions[col], errors='coerce')
    # Add a column for country (from the first column, which is always the country name)
    df_transactions['Country(2019)'] = df_transactions[df_transactions.columns[0]]
    dfs.append(df_transactions)

# Concatenate all country DataFrames
df_all = pd.concat(dfs, ignore_index=True)

# Set Country(2019) as index
df_all.set_index('Country(2019)', inplace=True)


# Convert all columns except 'Transactions(down)/Commodity(right)' to numeric, then divide only numeric columns by 3.6
for col in df_all.columns:
    if col != 'Transactions(down)/Commodity(right)':
        df_all[col] = pd.to_numeric(df_all[col], errors='coerce')

numeric_cols = df_all.columns.difference(['Transactions(down)/Commodity(right)'])
df_all[numeric_cols] = df_all[numeric_cols] / 0.0036 # Convert from TJ to MWh

# Save or display the final DataFrame
# print(df_all)

df_all.drop(columns= "Country (2019)", inplace=True)  # Drop rows where all values are NaN
# After cleaning/conversion

C:\Users\eduque\AppData\Local\Temp\ipykernel_31296\880164840.py:29: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace('---', np.nan)
C:\Users\eduque\AppData\Local\Temp\ipykernel_31296\880164840.py:30: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.lstrip('*') if isinstance(x, str) else x)
C:\Users\eduque\AppData\Local\Temp\ipykernel_31296\880164840.py:29: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace('---', np.nan)
C:\Users\eduque\AppData

In [57]:
df_all.loc["Chile",["Transactions(down)/Commodity(right)","Total Energy"]].sum()

Transactions(down)/Commodity(right)    Iron and steelChemical and petrochemicalNon-fe...
Total Energy                                                            122465277.777778
dtype: object

In [58]:
df_morocco = df_all.loc["Chile",["Transactions(down)/Commodity(right)","Total Energy"]]

total = df_morocco["Total Energy"].sum()
df_morocco["share_of_total"] = df_morocco["Total Energy"] / total
df_morocco

# ["Egypt",
# "South Africa",
# "Morocco",
# "Chile",]

,Transactions(down)/Commodity(right),Total Energy,share_of_total
Country(2019),,,
Chile,Iron and steel,1.940833e+06,0.015848
Chile,Chemical and petrochemical,1.684444e+06,0.013754
Chile,Non-ferrous metals,NaN,NaN
Chile,Non-metallic minerals,3.305278e+06,0.026990
Chile,Transport equipment,NaN,NaN
Chile,Machinery,NaN,NaN
Chile,Mining and quarrying,5.553111e+07,0.453444
Chile,Food and tobacco,6.197222e+05,0.005060
Chile,Paper pulp and printing,2.774056e+07,0.226518


In [ ]:
# Calculate the share per fuel of each transaction for South Africa (or any country)
country = ["Chile",
# "South Africa",
# "Morocco",
# "Chile",
]
# Exclude 'Transactions(down)/Commodity(right)', 'Total Energy', and 'memo: Of which Renewables' from fuel columns
exclude_cols = ["Transactions(down)/Commodity(right)", "Total Energy", "memo: Of which Renewables"]
fuel_cols = [col for col in df_all.columns if col not in exclude_cols]

# For each transaction, calculate the share per fuel
sa = df_all.loc[country]
shares = sa[fuel_cols].div(sa[fuel_cols].sum(axis=1), axis=0)
shares["Transactions(down)/Commodity(right)"] = sa["Transactions(down)/Commodity(right)"]
# Reorder columns to put transaction first
shares = shares[["Transactions(down)/Commodity(right)"] + fuel_cols]

# Add a column at the end to show the sum of shares (should be 1.0)
shares["share_sum"] = shares[fuel_cols].sum(axis=1)

# Display the shares DataFrame
shares.reset_index(drop=True, inplace=True)
shares

,Transactions(down)/Commodity(right),Primary Coal and Peat,Coal and Peat Products,Primary Oil,Oil Products,Natural Gas,Biofuels and Waste,Nuclear,Electricity,Heat,share_sum
0,Iron and steel,NaN,0.492057,NaN,0.049950,0.085874,NaN,NaN,0.372120,NaN,1.0
1,Chemical and petrochemical,NaN,NaN,NaN,0.000000,0.978562,NaN,NaN,0.021438,NaN,1.0
2,Non-ferrous metals,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,Non-metallic minerals,0.008320,NaN,NaN,0.769392,0.030759,0.000000,NaN,0.191529,NaN,1.0
4,Transport equipment,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
5,Machinery,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
6,Mining and quarrying,0.005207,0.000835,NaN,0.467436,0.041053,0.000585,NaN,0.484883,NaN,1.0
7,Food and tobacco,0.818019,0.052443,NaN,0.100852,0.023308,0.005379,NaN,NaN,NaN,1.0
8,Paper pulp and printing,NaN,NaN,NaN,0.098822,0.056946,0.637324,NaN,0.206907,NaN,1.0
9,Wood and wood products,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


: 